# Create MERlin scripts

Generates the per-experiment MERlin config/run files into `SAMPLE_DIR/merlin/` — replacing the old shared cluster location (`~/Software/merfish-parameters/`). Codebook and microscope-parameters files are NOT copied here: they're shared reference data shipped in `MERci/data/configs/merlin/{codebooks,microscope}/`, and since this `MERci/` clone already lives inside `SAMPLE_DIR/`, the slurm script below references them by their path inside this clone directly — self-contained, no separate cluster-side copy step.

Run this after notebook 05 (needs `experiment_info.yaml`).

In [1]:
import os
import sys
from pathlib import Path
import pandas as pd

MERCI_DIR    = Path(os.getcwd()).parent.parent.parent.parent   # MERci/ (notebook lives in MERci/notebooks/prepare_imaging/<variant>/<acquisition>/)
SAMPLE_DIR   = MERCI_DIR.parent                  # experiment root, e.g. LT048_sample_26/
METADATA_DIR = SAMPLE_DIR / "metadata"
POSITIONS_DIR = SAMPLE_DIR / "positions"
MERLIN_DIR   = SAMPLE_DIR / "merlin"             # new per-experiment MERlin folder
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.experiment_info import load_experiment_info
from MERci.acquisition.merlin_config import (
    resolve_codebook_filename, resolve_microscope_parameters_filename,
    MerlinAnalysisSpec, create_merlin_analysis_parameters,
    create_cluster_resource_allocation, create_snakemake_parameters,
    create_slurm_submit_script,
)

info = load_experiment_info(METADATA_DIR / "experiment_info.yaml")
SAMPLE_NAME = info.sample_name
MICROSCOPE  = info.microscope

print(f"SAMPLE_NAME : {SAMPLE_NAME}")
print(f"MICROSCOPE  : {MICROSCOPE}")
print(f"lib_name    : {info.lib_name}")
print(f"data_home   : {info.data_home}")
print(f"merlin_home : {info.merlin_home}")
print(f"folder_name : {info.folder_name}")

SAMPLE_NAME : 251225_LT027_saving_time
MICROSCOPE  : ST2
lib_name    : LT2
data_home   : /n/holylfs06/LABS/zhuang_lab/Lab/shared/Leonardo/projects/lineage_tracing/experiments
merlin_home : /n/holylfs06/LABS/zhuang_lab/Lab/shared/Leonardo/projects/lineage_tracing/experiments/251225_LT027_saving_time/merlin
folder_name : 251225_LT027_saving_time/merfish/data


## Resolve shared reference files (codebook, microscope parameters)

These live in `MERci/data/configs/merlin/` — shipped with the repo, not regenerated per experiment. As a sanity check, the codebook's own `bit_names` row is compared against this experiment's bit count from `round_bit_color_map.csv`, to catch a mismatched codebook before it reaches the cluster.

In [2]:
codebook_path   = MERCI_DIR / "data" / "configs" / "merlin" / "codebooks" / resolve_codebook_filename(info.lib_name)
microscope_path = MERCI_DIR / "data" / "configs" / "merlin" / "microscope" / resolve_microscope_parameters_filename(MICROSCOPE)

for p in (codebook_path, microscope_path):
    if not p.exists():
        raise FileNotFoundError(
            f"{p} not found -- add it to MERci/data/configs/merlin/ before running this notebook."
        )

rbc_path = METADATA_DIR / "round_bit_color_map.csv"
if rbc_path.exists():
    n_bits = int(pd.read_csv(rbc_path)["round"].max())
    codebook_bit_names_line = next(
        line for line in codebook_path.read_text().splitlines() if line.startswith("bit_names")
    )
    codebook_n_bits = len(codebook_bit_names_line.split(",")) - 1
    if codebook_n_bits != n_bits:
        print(f"WARNING: codebook {codebook_path.name} has {codebook_n_bits} bits, "
              f"but this experiment images {n_bits} -- double-check lib_name/codebook choice.")
    else:
        print(f"Bit count OK: {n_bits} bits, matching {codebook_path.name}.")
else:
    n_bits = None
    print(f"WARNING: {rbc_path} not found -- run notebook 03 first.")

print(f"Codebook  : {codebook_path}")
print(f"Microscope: {microscope_path}")

Codebook  : c:\Users\Leonardo\Dropbox\research\analysis\LineageTracing\251225_LT027_saving_time\MERci\data\configs\merlin\codebooks\LT2v0_codebook.csv
Microscope: c:\Users\Leonardo\Dropbox\research\analysis\LineageTracing\251225_LT027_saving_time\MERci\data\configs\merlin\microscope\STORM2FUSION_2304_60xSil.json


## MERlin analysis-parameters JSON

Built from a compact `MerlinAnalysisSpec` (which steps to include) instead of copying and hand-editing a prior experiment's file. Adjust the spec below to match this experiment; see `MerlinAnalysisSpec`'s docstring for every tunable field.

In [3]:
spec = MerlinAnalysisSpec(
    n_optimize_iterations = n_bits or 15,
    include_reporting     = True,
    include_segmentation  = False,   # True to append a cell-segmentation chain
    segmentation_method   = "CellPoseSegment3D",   # or "CellPoseSegmentSAM"
)

analysis_path = MERLIN_DIR / "analysis" / f"merlin_analysis_{SAMPLE_NAME}.json"
create_merlin_analysis_parameters(spec, analysis_path)
print(f"Saved: {analysis_path}")

Saved: c:\Users\Leonardo\Dropbox\research\analysis\LineageTracing\251225_LT027_saving_time\merlin\analysis\merlin_analysis_251225_LT027_saving_time.json


## Snakemake cluster-resource-allocation + parameters JSON

In [4]:
cluster_template = MERCI_DIR / "data" / "configs" / "merlin" / "snakemake" / "cluster_resource_allocation_basic.json"

cluster_resource_path = MERLIN_DIR / "snakemake" / f"cluster_resource_allocation_{SAMPLE_NAME}.json"
create_cluster_resource_allocation(
    template_path=cluster_template, exp_name=SAMPLE_NAME,
    n_optimize_iterations=spec.n_optimize_iterations, output_path=cluster_resource_path,
)
print(f"Saved: {cluster_resource_path}")

snakemake_params_path = MERLIN_DIR / "snakemake" / f"parameters_{SAMPLE_NAME}.json"
create_snakemake_parameters(
    exp_name=SAMPLE_NAME, cluster_config_path=cluster_resource_path,
    output_path=snakemake_params_path,
)
print(f"Saved: {snakemake_params_path}")

Saved: c:\Users\Leonardo\Dropbox\research\analysis\LineageTracing\251225_LT027_saving_time\merlin\snakemake\cluster_resource_allocation_251225_LT027_saving_time.json
Saved: c:\Users\Leonardo\Dropbox\research\analysis\LineageTracing\251225_LT027_saving_time\merlin\snakemake\parameters_251225_LT027_saving_time.json


## Slurm submit script

References the data-organization CSV (notebook 04, in `metadata/`) and positions file (notebook 02, in `positions/`) by their existing paths -- neither is duplicated into `merlin/`.

In [5]:
data_org_path = METADATA_DIR / f"data_organization_{MICROSCOPE.upper()}_{SAMPLE_NAME}.csv"
positions_path = POSITIONS_DIR / f"positions_{SAMPLE_NAME}.txt"
for p in (data_org_path, positions_path):
    if not p.exists():
        print(f"WARNING: {p} not found. For a multi-boundary experiment, set the "
              f"correct per-segment file path manually before submitting to the cluster.")

submit_path = MERLIN_DIR / "slurm" / "submit" / f"merlin_slurm_{SAMPLE_NAME}.sh"
create_slurm_submit_script(
    label                   = SAMPLE_NAME,
    parameters_file         = str(snakemake_params_path),
    analysis_file           = str(analysis_path),
    data_organization_file  = str(data_org_path),
    positions_file          = str(positions_path),
    codebook_file           = str(codebook_path),
    microscope_file         = str(microscope_path),
    data_home               = info.data_home,
    merlin_home             = info.merlin_home,
    folder_name             = info.folder_name,
    output_path             = submit_path,
)
print(f"Saved: {submit_path}\n")
print(submit_path.read_text())

Saved: c:\Users\Leonardo\Dropbox\research\analysis\LineageTracing\251225_LT027_saving_time\merlin\slurm\submit\merlin_slurm_251225_LT027_saving_time.sh

#!/bin/bash
#SBATCH -n 1
#SBATCH -N 1
#SBATCH -p zhuang
#SBATCH -t 2-00:00:00
#SBATCH --mem 5000
#SBATCH --open-mode=append
#SBATCH -o c:\Users\Leonardo\Dropbox\research\analysis\LineageTracing\251225_LT027_saving_time\merlin\slurm\out\251225_LT027_saving_time.out
#SBATCH -e c:\Users\Leonardo\Dropbox\research\analysis\LineageTracing\251225_LT027_saving_time\merlin\slurm\err\251225_LT027_saving_time.err

date +'Starting at %R.'

module load cuda/12.9.1-fasrc01
module load cudnn/9.10.2.21_cuda12-fasrc01
module load python
export CONDA_PKGS_DIRS=/n/holylabs/zhuang_lab/Lab/lsepulvedaduran/conda/pkgs
export CONDA_ENVS_PATH=/n/holylabs/zhuang_lab/Lab/lsepulvedaduran/conda/envs
source activate merlin_cp4_env
echo 251225_LT027_saving_time

merlin -k c:\Users\Leonardo\Dropbox\research\analysis\LineageTracing\251225_LT027_saving_time\merlin\snak